# Baseline and logistic regression — Tokke–Vinje

This standalone notebook reads raw competition data. It compares a smoothed generator × local hour × season frequency baseline with 14 regularized logistic regressions. Validation uses expanding windows for 2020, 2021 and 2022 with a seven-day gap. All probabilities are scored together with micro ROC-AUC.

Run all cells from the repository root or `notebooks/`. Predictions go to `outputs/baseline/` and `outputs/logistic_regression/`; shared evaluation goes to `outputs/evaluation/`. No future reservoir volumes or calculation times are used. Full-horizon prices and inflows and terminal water values are assumed to be allowed scheduling inputs; verify this assumption against the competition instructions before submission.

Hyperparameters are initial choices, not tuned results: L2 regularization C=1, baseline shrinkage=100 observations, and a maximum of 300 solver iterations. Document subsequent changes and evaluate them on the same splits. No class balancing or per-generator rank normalization is applied.


In [1]:
import os
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "4"
import json, re, warnings, time
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.exceptions import ConvergenceWarning

SEED = 71
H = 168
VALIDATION_YEARS = (2020, 2021, 2022)
ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "data/kernel").is_dir())
OUT = ROOT / "outputs"
MODEL_DIRS = {name: OUT / name for name in ("baseline", "logistic_regression")}
EVAL_DIR = OUT / "evaluation"
for directory in [*MODEL_DIRS.values(), EVAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
KERNEL = ROOT / "data/kernel"
EXTENDED = ROOT / "data/extended"


## Load and validate raw data

In [2]:
def read_series(path):
    frame = pd.read_csv(path)
    frame["date"] = pd.to_datetime(frame["date"], utc=True)
    frame = frame.set_index("date").sort_index()
    assert frame.index.is_unique and not frame.isna().any().any(), path
    return frame

uc = pd.read_csv(KERNEL / "Unit_commitment_decisions.csv")
test_map = pd.read_csv(ROOT / "prediction_mapping.csv")
sample = pd.read_csv(ROOT / "sample_submission.csv")
price_raw = read_series(KERNEL / "Historical_day_ahead_price_2015_2025.csv")
inflow = read_series(KERNEL / "Historical_inflow_1958_2025.csv")
volume = read_series(KERNEL / "Historical_volume_2015_2024.csv")
water_value = read_series(KERNEL / "Synthetic_water_value_2015_2024.csv")
with (EXTENDED / "Tokke_Vinje_topology.yaml").open() as handle:
    topology = yaml.safe_load(handle)
assert "Day-ahead price (EUR/MWh)" in price_raw
price = price_raw["Day-ahead price (EUR/MWh)"].resample("h").mean()
label_cols = [c for c in sample if c != "Run No"]
UNITS = list(dict.fromkeys(re.fullmatch(r"result_committed_(.+)_t\d+", c).group(1) for c in label_cols))
assert len(UNITS) == 14
assert label_cols == [f"result_committed_{u}_t{t}" for u in UNITS for t in range(H)]
uc = uc.sort_values("Run No").reset_index(drop=True)
cases = pd.concat([uc[["Run No", "starttime"]].assign(split="train"),
                   test_map.assign(split="test")], ignore_index=True).sort_values("Run No").reset_index(drop=True)
cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True)
assert cases["Run No"].is_unique
assert set(test_map["Run No"]) == set(sample["Run No"])
Y = uc[label_cols].to_numpy(dtype=np.float32).reshape(len(uc), len(UNITS), H).transpose(0, 2, 1)
assert np.isin(Y, [0, 1]).all()
print(cases.groupby("split")["starttime"].agg(["min", "max", "count"]))
print("Training target shape:", Y.shape)


                            min  ... count
split                            ...      
test  2023-01-01 00:00:00+00:00  ...   725
train 2015-01-01 00:00:00+00:00  ...  2922

[2 rows x 3 columns]
Training target shape: (2922, 168, 14)


## Build features from available scheduling inputs

Calendar days are computed directly from timestamps. Initial storage and terminal water values are looked up exactly. Historical price features are omitted to avoid fabricated history for early cases. Equal prices receive equal average ranks. Reservoir mappings are derived from YAML connections.

In [3]:
connections = pd.DataFrame(topology["connections"])
def feeding_reservoirs(plant):
    pending, seen, found = [plant], set(), set()
    while pending:
        node = pending.pop()
        if node in seen:
            continue
        seen.add(node)
        for _, link in connections.loc[connections["to"] == node].iterrows():
            if link["from_type"] == "reservoir":
                found.add(link["from"])
            elif link["from_type"] in ("river", "tunnel"):
                pending.append(link["from"])
    assert found, plant
    return sorted(found)

UNIT_RESERVOIRS = {u: feeding_reservoirs(u.rsplit("_", 1)[0]) for u in UNITS}
starts = pd.DatetimeIndex(cases["starttime"])
timestamps = starts.tz_convert(None).to_numpy()[:, None] + np.arange(H)[None, :] * np.timedelta64(1, "h")
flat_times = pd.DatetimeIndex(timestamps.ravel()).tz_localize("UTC")
local = flat_times.tz_convert("Europe/Oslo")
P = price.reindex(flat_times).to_numpy(dtype=np.float32).reshape(-1, H)
assert np.isfinite(P).all(), "Missing horizon prices; do not silently fill."
rank = pd.DataFrame(P).rank(axis=1, method="average", pct=True).to_numpy(dtype=np.float32)
X = pd.DataFrame({
    "t": np.tile(np.arange(H), len(cases)),
    "hour": local.hour,
    "weekday": local.dayofweek,
    "season": (local.month - 1) // 3,
    "doy_sin": np.sin(2 * np.pi * local.dayofyear / 365.25),
    "doy_cos": np.cos(2 * np.pi * local.dayofyear / 365.25),
    "price": P.ravel(),
    "price_rank": rank.ravel(),
    "price_minus_mean": (P - P.mean(axis=1, keepdims=True)).ravel(),
    "price_week_mean": np.repeat(P.mean(axis=1), H),
})
for unit, reservoirs in UNIT_RESERVOIRS.items():
    terminal = water_value.loc[starts + pd.Timedelta(days=7), reservoirs].mean(axis=1).to_numpy(dtype=np.float32)
    initial_fill = np.mean([volume.loc[starts, r].to_numpy() / topology["model"]["reservoir"][r]["max_vol"] for r in reservoirs], axis=0)
    unit_inflow = inflow.reindex(flat_times)[reservoirs].sum(axis=1, min_count=len(reservoirs)).to_numpy(dtype=np.float32)
    X[f"wv_{unit}"] = np.repeat(terminal, H)
    X[f"fill_{unit}"] = np.repeat(initial_fill, H)
    X[f"inflow_{unit}"] = unit_inflow
    X[f"spread_{unit}"] = X["price"] - X[f"wv_{unit}"]
    X[f"rank_fill_{unit}"] = X["price_rank"] * X[f"fill_{unit}"]
    X[f"spread_t_{unit}"] = X[f"spread_{unit}"] * X["t"] / (H - 1)
X = X.astype(np.float32)
assert np.isfinite(X.to_numpy()).all()
assert np.array_equal(cases.loc[cases["split"] == "train", "Run No"], uc["Run No"])
print("Feature shape:", X.shape, "Memory (MB):", X.memory_usage(deep=True).sum() / 1e6)


Feature shape: (612696, 94) Memory (MB): 230.373828


## Baseline, logistic regression, and validation

The baseline shrinks each generator/hour/season frequency towards that generator’s training frequency. Both models fit only on the current training fold. Logistic regression uses generator-specific domain features and shared calendar/price features. Scaling and categorical encoding are fitted inside each pipeline. Coefficients refer to standardized numerical features and are associations, not causal effects.

In [4]:
COMMON = ["t", "hour", "weekday", "season", "doy_sin", "doy_cos", "price", "price_rank", "price_minus_mean", "price_week_mean"]
CATEGORICAL = ["hour", "weekday", "season"]
def columns_for(unit):
    return COMMON + [f"{prefix}_{unit}" for prefix in ("wv", "fill", "inflow", "spread", "rank_fill", "spread_t")]

def micro_auc(truth, probability):
    assert truth.shape == probability.shape
    assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
    return float(roc_auc_score(truth.reshape(-1), probability.reshape(-1)))

def fit_baseline(features, targets, alpha=100.0):
    keys = (features["season"].to_numpy().astype(int) * 24 + features["hour"].to_numpy().astype(int))
    prior = targets.mean(axis=0)
    counts = np.bincount(keys, minlength=96)
    sums = np.stack([np.bincount(keys, weights=targets[:, j], minlength=96) for j in range(len(UNITS))], axis=1)
    return (sums + alpha * prior) / (counts[:, None] + alpha)

def baseline_predict(table, features):
    keys = features["season"].to_numpy().astype(int) * 24 + features["hour"].to_numpy().astype(int)
    return table[keys].astype(np.float32)

def fit_logistic(features, targets):
    models = {}
    for j, unit in enumerate(UNITS):
        columns = columns_for(unit)
        numeric = [c for c in columns if c not in CATEGORICAL]
        transform = ColumnTransformer([
            ("numeric", StandardScaler(), numeric),
            ("categorical", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), CATEGORICAL),
        ])
        model = make_pipeline(transform, LogisticRegression(C=1.0, max_iter=300, solver="lbfgs", random_state=SEED))
        if np.unique(targets[:, j]).size < 2:
            models[unit] = float(targets[:, j].mean())
            continue
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(features[columns], targets[:, j])
        if any(issubclass(w.category, ConvergenceWarning) for w in caught):
            raise RuntimeError(f"{unit}: solver did not converge; increase max_iter before trusting results.")
        models[unit] = model
    return models

def logistic_predict(models, features):
    return np.column_stack([
        np.full(len(features), models[u]) if isinstance(models[u], float)
        else models[u].predict_proba(features[columns_for(u)])[:, 1]
        for u in UNITS
    ]).astype(np.float32)

def rows_for(case_indices):
    return (np.asarray(case_indices)[:, None] * H + np.arange(H)[None, :]).ravel()

results, coefficient_tables = [], []
for year in VALIDATION_YEARS:
    valid_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
    train_indices = cases.index[(cases["split"] == "train") & (cases["starttime"] < valid_start - pd.Timedelta(days=7))].to_numpy()
    valid_indices = cases.index[(cases["split"] == "train") & (cases["starttime"].dt.year == year)].to_numpy()
    assert cases.loc[train_indices, "starttime"].max() + pd.Timedelta(days=7) < cases.loc[valid_indices, "starttime"].min()
    xt, xv = X.iloc[rows_for(train_indices)], X.iloc[rows_for(valid_indices)]
    yt, yv = Y[train_indices].reshape(-1, len(UNITS)), Y[valid_indices].reshape(-1, len(UNITS))
    started = time.monotonic()
    baseline = baseline_predict(fit_baseline(xt, yt), xv)
    models = fit_logistic(xt, yt)
    logistic = logistic_predict(models, xv)
    for name, prediction in (("baseline", baseline), ("logistic_regression", logistic)):
        results.append({"model": name, "validation_year": year, "micro_auc": micro_auc(yv, prediction),
                        "train_cases": len(train_indices), "valid_cases": len(valid_indices)})
        ordered = prediction.reshape(-1, H, len(UNITS)).transpose(0, 2, 1).reshape(len(valid_indices), -1)
        validation_frame = pd.DataFrame(ordered, columns=label_cols)
        validation_frame.insert(0, "Run No", cases.loc[valid_indices, "Run No"].to_numpy())
        validation_frame.to_csv(MODEL_DIRS[name] / f"oof_{name}_{year}.csv", index=False)
    for unit, model in models.items():
        if isinstance(model, float):
            continue
        names = model[0].get_feature_names_out()
        coefficient_tables.append(pd.DataFrame({"validation_year": year, "unit": unit, "feature": names,
                                               "coefficient": model[1].coef_[0]}))
    print(year, results[-2:], "Elapsed seconds:", round(time.monotonic() - started, 1), flush=True)

scores = pd.DataFrame(results)
scores.to_csv(EVAL_DIR / "cv_scores.csv", index=False)
summary = scores.groupby("model")["micro_auc"].agg(["mean", "std", "min", "max"])
print(summary)
coefficients = pd.concat(coefficient_tables, ignore_index=True)
coefficients.to_csv(MODEL_DIRS["logistic_regression"] / "coefficients.csv", index=False)
coefficients.assign(magnitude=lambda d: d.coefficient.abs()).sort_values("magnitude", ascending=False).head(20)


2020 [{'model': 'baseline', 'validation_year': 2020, 'micro_auc': 0.5692819363192233, 'train_cases': 1819, 'valid_cases': 366}, {'model': 'logistic_regression', 'validation_year': 2020, 'micro_auc': 0.8467006993836317, 'train_cases': 1819, 'valid_cases': 366}] Elapsed seconds: 36.3
2021 [{'model': 'baseline', 'validation_year': 2021, 'micro_auc': 0.5131904591114719, 'train_cases': 2185, 'valid_cases': 365}, {'model': 'logistic_regression', 'validation_year': 2021, 'micro_auc': 0.904635245002101, 'train_cases': 2185, 'valid_cases': 365}] Elapsed seconds: 41.3
2022 [{'model': 'baseline', 'validation_year': 2022, 'micro_auc': 0.5731111252879606, 'train_cases': 2550, 'valid_cases': 365}, {'model': 'logistic_regression', 'validation_year': 2022, 'micro_auc': 0.943220997154711, 'train_cases': 2550, 'valid_cases': 365}] Elapsed seconds: 46.4
                         mean  ...       max
model                          ...          
baseline             0.551861  ...  0.573111
logistic_regressio

## Fit all training cases and export both predictions

These CSV files are candidates, not automatically submitted to Kaggle. Select models using the validation results. OOF CSV files use the same wide column layout as Kaggle submissions and preserve validation case IDs for comparisons with other models.

In [5]:
def to_submission(predictions, run_nos):
    ordered = predictions.reshape(-1, H, len(UNITS)).transpose(0, 2, 1).reshape(-1, len(label_cols))
    frame = pd.DataFrame(ordered, columns=label_cols)
    frame.insert(0, "Run No", run_nos)
    frame = frame.set_index("Run No").loc[sample["Run No"]].reset_index()
    assert frame.columns.equals(sample.columns) and frame.shape == sample.shape
    assert np.isfinite(frame[label_cols].to_numpy()).all()
    return frame

train_indices = cases.index[cases["split"] == "train"].to_numpy()
test_indices = cases.index[cases["split"] == "test"].to_numpy()
xt, xe = X.iloc[rows_for(train_indices)], X.iloc[rows_for(test_indices)]
yt = Y.reshape(-1, len(UNITS))
full_models = fit_logistic(xt, yt)
for name, prediction in (
    ("baseline", baseline_predict(fit_baseline(xt, yt), xe)),
    ("logistic_regression", logistic_predict(full_models, xe)),
):
    to_submission(prediction, cases.loc[test_indices, "Run No"].to_numpy()).to_csv(MODEL_DIRS[name] / f"submission_{name}.csv", index=False)

log = scores.copy()
log.insert(0, "date_utc", datetime.now(timezone.utc).isoformat())
log.insert(1, "who", "Group 71")
log["change"] = "Initial season/hour frequency baseline and L2 logistic regression; seven-day validation gap"
log["kaggle_score"] = np.nan
log.to_csv(EVAL_DIR / "experiment_log.csv", index=False)
(EVAL_DIR / "metadata.json").write_text(json.dumps({"seed": SEED, "validation_years": VALIDATION_YEARS,
    "gap_days": 7, "units": UNITS, "unit_reservoirs": UNIT_RESERVOIRS,
    "logistic_C": 1.0, "baseline_alpha": 100, "features": {u: columns_for(u) for u in UNITS}}, indent=2))
print("Artifacts written to", OUT)


Artifacts written to /home/jupyter/TDT4173-Group-71/outputs


## Interpretation and next experiment

Compare the year-specific scores before choosing a model. Standard deviation is descriptive across three years, not a confidence interval. Per-generator scores can aid diagnosis, but the competition score is the global micro-AUC. Inspect standardized coefficients and compare predicted and observed generator frequencies before considering calibration or ensembles.

Next hypothesis: generator-specific nonlinear price-rank and storage effects improve over linear relationships. Test one feature change at a time on these exact folds. This notebook supplies modeling and coefficient interpretation; it does not replace the required EDA or complete project report.
